# Séance 10 — Arbres et forêts

**Prévoir, suite — Analyse des données, L3 Économie**

La semaine dernière, nous avons prévu avec les moindres carrés. Aujourd'hui nous changeons **l'estimateur, et rien d'autre** : mêmes données, mêmes variables, même découpage, mêmes repères.

La question : *un modèle plus puissant trouve-t-il ce que la régression n'a pas trouvé ?*

> *Exécution → Tout exécuter* avant de démarrer.


## 0. Un mot sur l'apprentissage automatique

**Apprendre de l'expérience** plutôt que suivre des règles écrites à l'avance : on ne programme pas la réponse, on fournit des exemples et l'algorithme en tire une règle.

L'**intelligence artificielle** est le grand ensemble ; l'**apprentissage automatique** (*machine learning*) en est la partie qui apprend des données ; l'**apprentissage profond** (*deep learning*) en est la partie qui le fait avec des réseaux de neurones à plusieurs couches. Nous restons au deuxième niveau.

Trois familles : **supervisé** (les exemples portent la bonne réponse — tout notre cours), **non supervisé** (pas de réponse : on cherche des groupes ou des axes), **par renforcement** (essais et récompenses).

> Une régression est déjà de l'apprentissage supervisé. Ce qui change aujourd'hui n'est pas la nature de l'exercice, c'est la **souplesse du modèle**.

### Les mêmes objets, deux vocabulaires

| | en statistique | en apprentissage automatique |
|---|---|---|
| ce qu'on explique | variable expliquée, endogène | **cible** (*target*, `y`) |
| ce qui explique | variables explicatives, régresseurs | **variables** (*features*, `X`) |
| une observation | un individu, une ligne | un *exemple* |
| estimer | estimer le modèle | **entraîner** (`fit`) |
| le résultat | des coefficients | un *modèle entraîné* |
| l'ajustement | $R^2$, résidus | **erreur** sur un jeu de test |

Les mots changent, les objets non. Une différence réelle, toutefois : la statistique veut un **coefficient juste**, l'apprentissage veut une **erreur faible**.

### La méthode, et les deux façons de se tromper

Quatre étapes, toujours les mêmes : **découper** → **entraîner** sur l'apprentissage → **prédire** sur le test → **mesurer l'erreur là**, et nulle part ailleurs.

Deux erreurs opposées. Un modèle **trop simple** rate la forme même sur les données d'apprentissage : erreur de **biais**. Un modèle **trop souple** passe par chaque point, donc apprend le bruit : erreur de **variance**. Les deux prédisent mal.


## 1. Les données

Deux colonnes seulement — le cours de clôture du titre et celui de l'indice. **Une ligne = une séance de bourse**, environ 252 par an : ni les week-ends, ni les jours fériés.


In [ ]:
!pip install -q yfinance

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import RandomForestRegressor
import yfinance as yf

TITRE, MARCHE = "RNO.PA", "^FCHI"

brut = yf.download([TITRE, MARCHE], start="2019-01-01", end="2025-12-31",
                   auto_adjust=True, progress=False)
px  = brut["Close"].dropna()
vlm = brut["Volume"].dropna()
px.columns  = ["MARCHE" if c == MARCHE else "TITRE" for c in px.columns]
vlm.columns = ["MARCHE" if c == MARCHE else "TITRE" for c in vlm.columns]

print(len(px), "seances, du", px.index.min().date(), "au", px.index.max().date())
print("soit", round(len(px) / ((px.index.max() - px.index.min()).days / 365.25)),
      "seances par an\n")
px.head(3).round(2)

## 2. De deux colonnes à onze variables

Tout le reste est **construit**. Et chaque variable porte un `.shift(1)` : elle doit être connue **la veille au soir**.


In [ ]:
r = np.log(px).diff()

X = pd.DataFrame(index=px.index)
for k in range(1, 6):
    X[f"r_lag{k}"] = r["TITRE"].shift(k)          # les cinq derniers rendements
X["ma5"]     = r["TITRE"].shift(1).rolling(5).mean()
X["ma21"]    = r["TITRE"].shift(1).rolling(21).mean()
X["vol5"]    = r["TITRE"].shift(1).rolling(5).std()
X["vol21"]   = r["TITRE"].shift(1).rolling(21).std()
X["marche1"] = r["MARCHE"].shift(1)
X["lvolume"] = np.log(vlm["TITRE"]).shift(1)
VARS = list(X.columns)

print(len(VARS), "variables\n")
X[["r_lag1", "r_lag2", "ma5", "vol21", "marche1", "lvolume"]].dropna().head(3).round(4)

In [ ]:
# combien de lignes coute la construction ?
d = pd.concat([r["TITRE"].abs().rename("y"), X], axis=1)
print("lignes au depart      :", len(d))
print("lignes completes      :", len(d.dropna()))
print("perdues               :", len(d) - len(d.dropna()),
      "  (il faut 21 jours pour une moyenne sur 21 jours)")
print()
print(X.describe().T[["count", "mean", "std", "min", "max"]].round(4).to_string())

| nom | nombre | ce qu'elle capte |
|---|---|---|
| `r_lag1` … `r_lag5` | 5 | les cinq derniers rendements : une éventuelle tendance courte |
| `ma5`, `ma21` | 2 | la direction moyenne récente, lissée |
| `vol5`, `vol21` | 2 | l'agitation récente — le regroupement de volatilité |
| `marche1` | 1 | ce qu'a fait le CAC 40 la veille |
| `lvolume` | 1 | l'intensité des échanges, en logarithme |

**Deux cibles, le même tableau de variables :** le rendement `r` (signe compris) et son amplitude `|r|`.

## 3. Le protocole, inchangé


In [ ]:
def jeu(cible):
    d = pd.concat([cible.rename("y"), X], axis=1).dropna()
    c = int(len(d) * 0.8)
    return d.iloc[:c], d.iloc[c:]

def r2_hors(y, prev, repere):
    y = np.asarray(y)
    return 1 - np.sum((y - prev)**2) / np.sum((y - repere)**2)

tr, te = jeu(r["TITRE"].abs())          # cible : l'amplitude
print("apprentissage :", tr.index[0].date(), "->", tr.index[-1].date(), f"({len(tr)})")
print("test          :", te.index[0].date(), "->", te.index[-1].date(), f"({len(te)})")

Si les résultats diffèrent aujourd'hui, **ce ne peut être que le modèle**. Aucune autre explication n'est disponible.

## 4. L'arbre : une constante par région

Pour voir l'idée, un arbre sur **une seule variable** : la volatilité récente.


## 3. Un arbre à la main, sur dix journées

Avant de lancer la machine sur 1 382 jours et onze variables, on fait pousser un arbre **à la main**, sur dix journées et une seule variable. Tout ce que fait un arbre de décision tient dans ce qui suit.

On tire dix journées du fichier — deux dans chaque cinquième de volatilité, pour que les dix points couvrent toute la plage — et on ne garde que deux colonnes : la volatilité des 21 derniers jours (ce qu'on sait le matin) et l'amplitude du lendemain (ce qu'on cherche à prévoir).

In [ ]:
q   = pd.qcut(tr["vol21"], 5, labels=False)
dix = pd.concat([g.sample(2, random_state=4)
                 for _, g in tr.groupby(q)]).sort_values("vol21")

vol  = (dix["vol21"] * 100).round(1).values      # en %
ampl = (dix["y"]     * 100).round(1).values      # en %

pd.DataFrame({"volatilité (%)": vol, "amplitude du lendemain (%)": ampl},
             index=dix.index.date).T

**Première question : si vous ne pouvez donner qu'*une seule* prévision pour les dix jours, laquelle ?**

La moyenne — parce que c'est le nombre qui minimise la somme des carrés des écarts. C'est la même propriété qu'en régression. La somme obtenue est **l'erreur à battre**.

In [ ]:
def erreur(y):                      # somme des carres des ecarts a la moyenne
    return ((y - y.mean()) ** 2).sum()

print("moyenne         :", round(ampl.mean(), 2), "%")
print("erreur a battre :", round(erreur(ampl), 2))

**Deuxième question : et si on avait le droit de couper l'axe en deux ?**

On prédirait alors la moyenne de chaque côté. L'arbre essaie *tous* les seuils possibles — ici il y en a six — et garde celui qui laisse le moins d'erreur. C'est la totalité de l'algorithme.

In [ ]:
u      = np.unique(vol)
seuils = (u[:-1] + u[1:]) / 2

lignes = []
for s in seuils:
    g, d = ampl[vol <= s], ampl[vol > s]
    lignes.append([s, len(g), round(g.mean(), 2),
                      len(d), round(d.mean(), 2),
                   round(erreur(g) + erreur(d), 2)])

pd.DataFrame(lignes, columns=["seuil", "n gauche", "moyenne gauche",
                              "n droite", "moyenne droite", "erreur restante"])

Le meilleur seuil fait tomber l'erreur d'environ **70 %**, et aucun autre n'en approche.

**Troisième question : est-ce bien ce que fait `scikit-learn` ?** `max_depth=1` lui interdit plus d'une coupure. Les deux feuilles qu'il produit sont, au centième près, les deux moyennes qu'on vient de calculer.

In [ ]:
a = DecisionTreeRegressor(max_depth=1).fit(vol.reshape(-1, 1), ampl)
print(export_text(a, feature_names=["vol21"]))

a.predict([[1.2], [2.5]])          # une journee calme, une journee agitee

Avec `max_depth=2`, il recommencerait **dans chaque moitié** : quatre feuilles, quatre moyennes. Rien de plus que ce qu'on vient de faire à la main.

Tout ce qui suit est ce même calcul, répété sur 1 382 jours et onze variables.

In [ ]:
t1 = DecisionTreeRegressor(max_depth=2, min_samples_leaf=80,
                           random_state=0).fit(tr[["vol21"]].values, tr["y"])

grille = np.linspace(tr["vol21"].min(), tr["vol21"].quantile(.995), 600)

fig, ax = plt.subplots(figsize=(7.5, 3))
ax.scatter(tr["vol21"] * 100, tr["y"] * 100, s=5, alpha=.20,
           color="#9AA0A0", linewidths=0)
ax.plot(grille * 100, t1.predict(grille.reshape(-1, 1)) * 100,
        color="#E2162F", lw=2.6, drawstyle="steps-post")
ax.set_xlabel("Volatilité des 21 derniers jours (%)")
ax.set_ylabel("|rendement| du jour (%)")
ax.set_ylim(0, tr["y"].quantile(.99) * 100)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Pas de droite, pas de coefficient : un **escalier**. L'arbre découpe l'axe en régions et prédit, dans chacune, la moyenne qu'il y observe.

## 5. L'arbre, en vrai


In [ ]:
t2 = DecisionTreeRegressor(max_depth=2, random_state=0).fit(tr[VARS], tr["y"])

fig, ax = plt.subplots(figsize=(11.5, 4))
plot_tree(t2, feature_names=VARS, filled=True, rounded=True, impurity=False,
          proportion=True, precision=4, fontsize=10, ax=ax)
plt.show()

In [ ]:
# les memes regles, en texte
print(export_text(t2, feature_names=VARS, decimals=4))

Il a choisi **tout seul** de commencer par la volatilité. Si l'agitation récente est faible, il prédit une petite amplitude ; forte, une grande. **Il a retrouvé le regroupement de volatilité sans qu'on le lui dise.**

## 6. Plus profond n'est pas mieux


In [ ]:
profondeurs = range(1, 17)
app, test = [], []
for p_ in profondeurs:
    t = DecisionTreeRegressor(max_depth=p_, random_state=0).fit(tr[VARS], tr["y"])
    app.append(t.score(tr[VARS], tr["y"]))
    test.append(r2_hors(te["y"], t.predict(te[VARS]), tr["y"].mean()))

fig, ax = plt.subplots(figsize=(7.8, 3))
ax.plot(profondeurs, app,  "o-", color="#3D4543", lw=2, ms=4)
ax.plot(profondeurs, test, "o-", color="#E2162F", lw=2, ms=4)
ax.text(16.4, app[-1],  "apprentissage", color="#3D4543", va="center")
ax.text(16.4, test[-1], "test",          color="#E2162F", va="center")
ax.axhline(0, color="#9AA0A0", ls=":", lw=1)
ax.set_xlabel("Profondeur de l'arbre"); ax.set_ylabel("$R^2$")
ax.set_xlim(.4, 19.6)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

meilleure = list(profondeurs)[int(np.argmax(test))]
print("meilleure profondeur :", meilleure, " R2 test =", round(max(test), 3))
print("a la profondeur 16   : apprentissage", round(app[-1], 3),
      "  test", round(test[-1], 3))

À la profondeur 16, l'arbre a presque **une feuille par journée** : il explique ce qu'on lui a montré et rien d'autre.

## 7. La forêt aléatoire

Un arbre seul est **instable** : retirez vingt journées et il se réorganise entièrement. Une forêt en fait pousser des centaines — chacun sur un tirage au sort des observations, et à chaque question un tirage au sort des variables — puis **moyenne** les prédictions. C'est la seule idée.


In [ ]:
# pourquoi moyenner : 20 arbres, chacun sur un tirage des donnees
scores = []
for i in range(20):
    ech = tr.sample(len(tr), replace=True, random_state=i)
    t = DecisionTreeRegressor(max_depth=8, random_state=0).fit(ech[VARS], ech["y"])
    scores.append(r2_hors(te["y"], t.predict(te[VARS]), tr["y"].mean()))

scores = np.array(scores)
print("20 arbres de profondeur 8, pris un par un :")
print("   R2 hors echantillon  min", round(scores.min(), 3),
      "  max", round(scores.max(), 3),
      "  moyenne", round(scores.mean(), 3))
print("   ecart-type entre arbres :", round(scores.std(), 3))

foret = RandomForestRegressor(n_estimators=300, max_depth=8,
                              random_state=0, n_jobs=-1).fit(tr[VARS], tr["y"])
print()
print("   la FORET faite des memes arbres :",
      round(r2_hors(te["y"], foret.predict(te[VARS]), tr["y"].mean()), 3))

Chaque arbre, pris isolément, est **inutilisable** : son $R^2$ hors échantillon est négatif, et il varie du simple au décuple selon le tirage. Leur **moyenne**, elle, devient positive.

C'est tout le mécanisme : les arbres se trompent beaucoup, mais pas tous dans le même sens. Moyenner annule une partie des erreurs et conserve ce qu'ils ont en commun — le signal.


## 7 bis. Le code, en entier

Tout ce qui précède tient en dix lignes. C'est la **régularité** de `scikit-learn` qu'il faut retenir : on crée le modèle, on appelle `.fit()`, puis `.score()` ou `.predict()`.

Changer d'algorithme, c'est changer **un seul mot**.


In [ ]:
from sklearn.tree     import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

app, test = jeu(r["TITRE"].abs())                 # 80 / 20 dans le temps
X_app,  y_app  = app[VARS],  app["y"]
X_test, y_test = test[VARS], test["y"]

arbre = DecisionTreeRegressor(max_depth=3)
arbre.fit(X_app, y_app)                           # .fit() ne voit JAMAIS le test

foret = RandomForestRegressor(n_estimators=300, min_samples_leaf=20)
foret.fit(X_app, y_app)

print("arbre :", round(arbre.score(X_test, y_test), 3))
print("foret :", round(foret.score(X_test, y_test), 3))

Pour prédire une **catégorie** plutôt qu'un nombre — hausse ou baisse, par exemple — on remplace `Regressor` par `Classifier`, et le reste est identique.

Attention : `.score()` ne renvoie pas la même chose dans les deux cas. Pour un *Regressor* c'est un $R^2$ (calculé contre la moyenne du **test**) ; pour un *Classifier*, c'est la part de prédictions correctes. Dans le cours nous utilisons `r2_hors`, qui prend pour repère la moyenne de l'**apprentissage** — la seule qu'on connaisse au moment de prévoir.


In [ ]:
resultats = []
for cible, nom in [(r["TITRE"], "rendement"), (r["TITRE"].abs(), "amplitude |r|")]:
    a, b = jeu(cible)
    modeles = {
        "regression":      LinearRegression(),
        "arbre (prof. 3)": DecisionTreeRegressor(max_depth=3, random_state=0),
        "foret":           RandomForestRegressor(n_estimators=300, min_samples_leaf=20,
                                                 random_state=0, n_jobs=-1),
    }
    for lab, mod in modeles.items():
        mod.fit(a[VARS], a["y"])
        resultats.append({
            "cible": nom, "modele": lab,
            "R2 apprentissage": round(mod.score(a[VARS], a["y"]), 3),
            "R2 hors echantillon": round(r2_hors(b["y"], mod.predict(b[VARS]),
                                                 a["y"].mean()), 3)})

print(pd.DataFrame(resultats).to_string(index=False))

Sur le **rendement**, la forêt apprend bien mieux que la régression — et prévoit *moins bien que rien*. Quand il n'y a pas de signal, un modèle plus souple ne trouve pas davantage : il surajuste plus vite.

Sur l'**amplitude**, où il y a du signal, les trois modèles se tiennent. La non-linéarité n'apporte rien ici, et c'est un résultat.

> **Ce n'est pas la puissance du modèle qui décide. C'est l'existence d'un signal.**

## 8. Les importances ne sont pas des coefficients


In [ ]:
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=20,
                           random_state=0, n_jobs=-1).fit(tr[VARS], tr["y"])
imp = pd.Series(rf.feature_importances_, index=VARS).sort_values()

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(imp.index, imp.values, color="#9AA0A0", edgecolor="white")
ax.barh(imp.index[-2:], imp.values[-2:], color="#65B32E", edgecolor="white")
ax.set_xlabel("Importance (part de la réduction d'erreur)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Une **importance** dit de combien une variable a réduit l'erreur dans la forêt. Elle n'a **ni signe, ni unité, ni écart-type** : impossible d'écrire « +1 point de volatilité ⇒ +x ».

Elle est biaisée vers les variables à nombreuses valeurs distinctes, et se **partage** entre variables corrélées — deux variables jumelles paraissent toutes deux secondaires.

**Ce qu'on perd :** pas d'intervalle de confiance, pas de test, pas d'effet marginal, aucune lecture causale. Une forêt prévoit bien et **n'explique rien**.

## 9. Ce qu'on retient des dix séances

1. Ce n'est pas la puissance du modèle qui décide : c'est l'existence d'un **signal**.
2. Le **protocole** ne change pas d'un modèle à l'autre — c'est lui qui rend une comparaison possible.
3. Un **échec de prévision bien mesuré** est un résultat, pas une erreur.
4. Et avant tout cela : savoir ce que décrit une ligne, ce que vaut un code de non-réponse, et ce qu'un coefficient ne dit pas.
